In [ ]:
def ensure_stage2_from_canonical(period_value: str, run_id_value: str):
    # Compatibility bridge while DFM adapters still write canonical_holdings.
    # Applies cash-mapping logic for rows with no ISIN/SEDOL.
    if not table_exists("canonical_holdings"):
        print("[bridge] canonical_holdings not found; skipping Stage 2 bridge")
        return

    source_df = spark.table("canonical_holdings").filter(
        (F.col("period") == period_value) & (F.col("run_id") == run_id_value)
    )

    if source_df.rdd.isEmpty():
        print("[bridge] no canonical_holdings rows to bridge")
        return

    # Cash mapping: if no ISIN/SEDOL, assign TPY_CASH_* code and Undertaking ID_type
    enriched = (
        source_df
        .withColumn(
            "_is_cash",
            F.col("isin").isNull() & F.col("security_id").isNull()
        )
        .withColumn(
            "security_code_out",
            F.when(
                F.col("_is_cash"),
                F.concat_ws("_", F.lit("TPY_CASH"), F.upper(F.coalesce(F.col("local_currency"), F.lit("XX"))))
            ).otherwise(F.col("security_id"))
        )
        .withColumn(
            "id_type_out",
            F.when(F.col("_is_cash"), F.lit("Undertaking - Specific"))
             .otherwise(F.col("id_type"))
        )
        .withColumn(
            "asset_name_out",
            F.when(F.col("_is_cash"), F.lit("CASH"))
             .otherwise(F.col("asset_name"))
        )
    )

    stage2 = (
        enriched
        .withColumn("profile_id", F.col("dfm_id"))
        .withColumn("row_hash", F.sha2(F.concat_ws("|", F.col("run_id"), F.col("dfm_id"), F.col("source_row_id")), 256))
        .withColumn("policyholder_number", F.col("policy_id"))
        .withColumn("security_code", F.col("security_code_out"))
        .withColumn("sedol", F.lit(None).cast("string"))
        .withColumn("include_flag", F.lit("Include"))
        .withColumn("exclusion_reason_code", F.lit(None).cast("string"))
        .withColumn("identifier_chosen", F.coalesce(F.col("security_code_out"), F.col("isin")))
        .withColumn(
            "decision_trace_json",
            F.to_json(
                F.struct(
                    F.lit("bridge_from_canonical_holdings").alias("strategy"),
                    F.col("dfm_id").alias("dfm_id"),
                    F.col("_is_cash").alias("cash_mapping_applied"),
                    F.current_timestamp().alias("bridged_at")
                )
            )
        )
        .select(
            "period",
            "run_id",
            "dfm_id",
            "profile_id",
            "source_file",
            "source_row_id",
            "row_hash",
            "policyholder_number",
            "security_code",
            "isin",
            "sedol",
            "other_security_id",
            F.col("id_type_out").alias("id_type"),
            F.col("asset_name_out").alias("asset_name"),
            F.col("holding").cast("decimal(28,8)").alias("holding"),
            F.col("local_bid_price").cast("decimal(28,8)").alias("local_bid_price"),
            "local_currency",
            F.col("fx_rate").cast("decimal(28,8)").alias("fx_rate"),
            F.col("cash_value_gbp").cast("decimal(28,8)").alias("cash_value_gbp"),
            F.col("bid_value_gbp").cast("decimal(28,8)").alias("bid_value_gbp"),
            F.col("accrued_interest_gbp").cast("decimal(28,8)").alias("accrued_interest_gbp"),
            "include_flag",
            "exclusion_reason_code",
            "identifier_chosen",
            "decision_trace_json",
            "data_quality_flags"
        )
        .dropDuplicates(["run_id", "dfm_id", "source_row_id"])
    )

    stage2.write.mode("append").saveAsTable("individual_dfm_consolidated")
    stage2_count = stage2.count()
    cash_count = stage2.filter(F.col("id_type") == "Undertaking - Specific").count()
    print(f"[bridge] appended {stage2_count} rows to individual_dfm_consolidated (including {cash_count} cash-mapped rows)")


StatementMeta(, 192f92f6-78bb-4563-be75-a95dda87885b, 3, Finished, Available, Finished, False)

[nb_run_all] START period=2026-03 run_id=20260302T220333Z
[orchestrator] RUNNING: Orchestration started

--- Running nb_ingest_wh_ireland with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_ingest_wh_ireland completed with result: NO_FILES

--- Running nb_ingest_pershing with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_ingest_pershing FAILED: Py4JJavaError: An error occurred while calling o5219.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].
---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 362
    330 good = good.withColumn(
    331     "source_row_id",
    332     F.sha2(
   (...)
    344     )
    345 )
    347 # ---------- Canonical projection ----------
    348 canonical = (
    349     good
    350     .withColumn("period", F.lit(period))
    351     .withColumn("run_id", F.lit(run_id))
    352     .withColumn("dfm_id", F.lit(dfm_id))
    353     .withColumn("dfm_name", F.lit(dfm_name))
    354     .withColumn("source_file", F.col("source_file_out"))
    355     .withColumn("source_sheet", F.lit(None).cast("string"))
    356     .wi

--- nb_ingest_castlebay FAILED: Py4JJavaError: An error occurred while calling o5229.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 85f41908-fd38-4b99-b667-d1fb8262c0c6).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- period: string (nullable = true)
-- run_id: string (nullable = true)
-- dfm_id: string (nullable = true)
-- source_file: string (nullable = true)
-- row_number: long (nullable = true)
-- column_name: string (nullable = true)
-- raw_value: string (nullable = true)
-- error_type: string (nullable = true)
-- error_message: string (nullable = true)
-- event_ts: timestamp (nul

--- nb_ingest_brown_shipley FAILED: Py4JJavaError: An error occurred while calling o5239.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].
---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 388
    355 good = good.withColumn(
    356     "source_row_id",
    357     F.sha2(
   (...)
    371     )
    372 )
    374 # ---------- Canonical projection ----------
    375 canonical = (
    376     good
    377     .withColumn("period", F.lit(period))
    378     .withColumn("run_id", F.lit(run_id))
    379     .withColumn("dfm_id", F.lit(dfm_id))
    380     .withColumn("dfm_name", F.lit(dfm_name))
    381     .withColumn("source_file", F.col("source_file_out"))
    382     .withColumn("source_sheet", F.lit(None).cast("string"))
    383   

--- nb_validate completed with result: NO_DATA

--- Running nb_aggregate with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_aggregate completed with result: NO_DATA

--- Running nb_reports with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_reports completed with result: NO_DATA
[orchestrator] PARTIAL: run_id=20260302T220333Z; ingest_failed=3/4; post_failed=0/3

=== RUN SUMMARY ===
run_id=20260302T220333Z; ingest_failed=3/4; post_failed=0/3
Ingest results:
('nb_ingest_wh_ireland', 'wh_ireland', 'OK', 'NO_FILES')
('nb_ingest_pershing', 'pershing', 'FAILED', 'Py4JJavaError: An error occurred while calling o5219.throwExceptionIfHave.\n: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].\n---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 362\n    330 good = good.withColumn(\n    331     "source_row_id",\n    332     F.sha2(\n   (...)\n    344     )\n    345 )\n    347 # ---------- Canonical projection ----------\n    348 canonical = (\n    349     good\n    350     .withColumn("period", F.l

In [ ]:
# 1) audit rows for this run
spark.sql(f"""
SELECT run_id, period, dfm_id, status, files_processed, rows_ingested, parse_errors_count, started_at, completed_at
FROM run_audit_log
WHERE run_id = '{run_id}'
ORDER BY dfm_id, completed_at
""").show(truncate=False)

# 2) stage 1 row counts by DFM
spark.sql(f"""
SELECT dfm_id, COUNT(*) AS rows
FROM source_dfm_raw
WHERE run_id = '{run_id}'
GROUP BY dfm_id
ORDER BY dfm_id
""").show()

# 3) stage 2 row counts by DFM
spark.sql(f"""
SELECT dfm_id, COUNT(*) AS rows
FROM individual_dfm_consolidated
WHERE run_id = '{run_id}'
GROUP BY dfm_id
ORDER BY dfm_id
""").show()

# 4) stage 3 row counts by DFM
spark.sql(f"""
SELECT dfm_id, COUNT(*) AS rows
FROM aggregated_dfms_consolidated
WHERE run_id = '{run_id}'
GROUP BY dfm_id
ORDER BY dfm_id
""").show()

StatementMeta(, 192f92f6-78bb-4563-be75-a95dda87885b, -1, Cancelled, , Cancelled, True)